In [0]:
from pathlib import Path
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql import Window

In [0]:
%sql
DROP DATABASE IF EXISTS workspace.silver CASCADE; 

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS workspace.silver
COMMENT 'Capa Silver procesados'

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.silver.predios_registro")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.silver.predios_registro (
    pk STRING,
    matricula STRING,
    fecha_radica DATE,
    fecha_apertura DATE,
    year_radica INT,
    orip INT,
    divipola INT,
    departamento_descripcion STRING,
    municipio_descripcion STRING,
    tipo_predio_descripcion STRING,
    categoria_ruralidad_descripcion STRING,
    num_anotacion INT,
    dinamica_2024 INT,
    natujur_id INT,
    natujur_descripcion STRING,
    documento_justificativo STRING,
    count_a INT,
    count_de INT,
    predios_nuevos INT,
    tiene_valor BOOLEAN,
    tiene_mas_de_un_valor BOOLEAN,
    estado_folio STRING,
    valor DOUBLE
)
USING DELTA
""")

In [0]:
# Leer bronze y pasar a pandas
df_spark = spark.table("workspace.bronze.predios_registro")

In [0]:
# Casteo
df_spark = df_spark.withColumn("year_radica", F.col("year_radica").cast("int")) \
                   .withColumn("orip", F.col("orip").cast("int")) \
                   .withColumn("divipola", F.col("divipola").cast("int")) \
                   .withColumn("num_anotacion", F.col("num_anotacion").cast("int")) \
                   .withColumn("dinamica_2024", F.col("dinamica_2024").cast("int")) \
                   .withColumn("cod_natujur", F.col("cod_natujur").cast("int")) \
                   .withColumn("count_a", F.col("count_a").cast("int")) \
                   .withColumn("count_de", F.col("count_de").cast("int")) \
                   .withColumn("predios_nuevos", F.col("predios_nuevos").cast("int")) \
                   .withColumn("tiene_valor", F.col("tiene_valor").cast("boolean")) \
                   .withColumn("tiene_mas_de_un_valor", F.col("tiene_mas_de_un_valor").cast("boolean")) \
                   .withColumn("valor", F.col("valor").cast("double"))

In [0]:

# 1. Extraer valores únicos (¡Asegúrate de usar el DF correcto para cada uno!)
df_deps_uniq = df_spark.select("departamento").distinct()
df_muns_uniq = df_spark.select("municipio").distinct()
df_pred_uniq = df_spark.select("tipo_predio_zona").distinct()
df_rurl_uniq = df_spark.select("categoria_ruralidad_2024").distinct()

# 2. Asignar IDs (Corrigiendo el origen y el nombre de la columna ID)
df_dim_dep = df_deps_uniq.withColumn("id_departamento", F.row_number().over(Window.orderBy("departamento")))
df_dim_mun = df_muns_uniq.withColumn("id_municipio", F.row_number().over(Window.orderBy("municipio")))
df_dim_tip = df_pred_uniq.withColumn("id_tipo_predio", F.row_number().over(Window.orderBy("tipo_predio_zona")))
df_dim_rur = df_rurl_uniq.withColumn("id_ruralidad", F.row_number().over(Window.orderBy("categoria_ruralidad_2024")))

# 3. Joins encadenados
# Tip: Usamos un alias o seleccionamos solo lo necesario para evitar columnas duplicadas
df_silver = df_spark \
    .join(df_dim_dep, "departamento", "left") \
    .join(df_dim_mun, "municipio", "left") \
    .join(df_dim_tip, "tipo_predio_zona", "left") \
    .join(df_dim_rur, "categoria_ruralidad_2024", "left")

In [0]:
from pyspark.sql.functions import to_date

df_silver = df_silver.withColumn(
    "fecha_radica",
    F.coalesce(
        F.try_to_date("fecha_radica_texto", "yyyy-MM-dd HH:mm:ss"),
        F.try_to_date("fecha_radica_texto", "yyyy-MM-dd"),
        F.try_to_date("fecha_radica_texto", "dd/MM/yyyy"),
        F.try_to_date("fecha_radica_texto", "dd/MM/yy")
    )
)

# Convertir fecha_apertura con varios formatos
df_silver = df_silver.withColumn(
    "fecha_apertura",
    F.coalesce(
        F.to_date("fecha_apertura_texto", "yyyy-MM-dd HH:mm:ss"),
        F.to_date("fecha_apertura_texto", "yyyy-MM-dd"),
        F.to_date("fecha_apertura_texto", "dd/MM/yyyy"),
        F.to_date("fecha_apertura_texto", "dd/MM/yy")
    )
)

df_silver = df_silver.withColumn(
    "categoria_ruralidad_descripcion",
    F.upper(F.col("categoria_ruralidad_descripcion"))
)

In [0]:
rename_map = {
    "pk": "pk",
    "matricula": "matricula",
    "fecha_radica": "fecha_radica",
    "fecha_apertura": "fecha_apertura",
    "year_radica": "year_radica",
    "orip": "orip",
    "divipola": "divipola",
    "departamento": "departamento_descripcion",
    "municipio": "municipio_descripcion",
    "tipo_predio_zona": "tipo_predio_descripcion",
    "categoria_ruralidad_2024": "categoria_ruralidad_descripcion",
    "num_anotacion": "num_anotacion",
    "dinamica_2024": "dinamica_2024",
    "cod_natujur": "natujur_id",
    "nombre_natujur": "natujur_descripcion",
    "documento_justificativo": "documento_justificativo",
    "count_a": "count_a",
    "count_de": "count_de",
    "predios_nuevos": "predios_nuevos",
    "tiene_valor": "tiene_valor",
    "tiene_mas_de_un_valor": "tiene_mas_de_un_valor",
    "estado_folio": "estado_folio",
    "valor": "valor",
}

# Aplicar el renombramiento
df = df.rename(columns=rename_map, errors="ignore")


In [0]:
# 6) Quitar duplicados
df = df.drop_duplicates()

In [0]:
# 8) Eliminar columnas duplicadas (si las hubiera)
df = df.loc[:, ~df.columns.duplicated()]

# 9) Regresar a Spark DataFrame
df_silver = spark.createDataFrame(df)


In [0]:
df_silver = df_silver.drop("folios_derivados")

In [0]:
# Castear columnas a los tipos correctos para que coincidan con el esquema de la tabla
df_silver = df_silver.withColumn("year_radica", F.col("year_radica").cast("int")) \
                     .withColumn("orip", F.col("orip").cast("int")) \
                     .withColumn("divipola", F.col("divipola").cast("int")) \
                     .withColumn("tiene_valor", F.col("tiene_valor").cast("boolean")) \
                     .withColumn("tiene_mas_de_un_valor", F.col("tiene_mas_de_un_valor").cast("boolean"))

# Usamos overwrite para reemplazar completamente los datos existentes
df_silver.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.predios_registro")